In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import dill
import json
import sys
import os
from scipy import spatial
import matplotlib.pyplot as plt

In [ ]:
dict_path = 'reshaped_ev_points_dict.csv'

In [ ]:
ev_points = pd.read_csv(os.path.join(result_path,dict_path))
cluster = np.unique(ev_points['cluster'].values)

#rand_clusters = np.random.choice(cluster, size=, replace=False)
rand_clusters = [71]

clusters_df = ev_points[ev_points['cluster'].isin(rand_clusters)]
ev_points_n_clusters = clusters_df[['ev_points_id', 'x', 'y', 'z']].values

In [ ]:
concentration_data_clusters =[]
for i in range(500):
    file_path = result_path + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    concentration_data_clusters.append(concentrations)

In [ ]:
frames = []
for i in range(500):
    frame = go.Frame(
        data=[
            go.Scatter3d(
                x=ev_points_n_clusters[:, 1],
                y=ev_points_n_clusters[:, 2],
                z=ev_points_n_clusters[:, 3],
                text = ev_points_n_clusters[:,0],
                mode='markers',
                marker=dict(
                    size=3,
                    color=concentration_data_clusters[i],  # Set color to the log-transformed concentrations
                    colorscale='Viridis',  # Color scale
                    cmin=0,  # Set minimum value for the color scale
                    cmax=np.log10(conc_max + epsilon),  # Set maximum value for the color scale
                    colorbar=dict(title='Log10 Concentration'),
                    opacity=0.8
                )
            )
        ],
        name=str(i)
    )
    frames.append(frame)

# Create the initial figure
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=ev_points_n_clusters[:, 1],
            y=ev_points_n_clusters[:, 2],
            z=ev_points_n_clusters[:, 3],
            text = ev_points_n_clusters[:,0],
            mode='markers',
            marker=dict(
                size=3,
                color=np.zeros(len(ev_points_n_clusters)),  # Default color (zeros)
                colorscale='Viridis',
                cmin=0,  # Set minimum value for the color scale
                cmax=np.log10(conc_max + epsilon),  # Set maximum value for the color scale
                colorbar=dict(title='Log10 Concentration'),
                opacity=0.8
            )
        )
    ],
    layout=go.Layout(
        updatemenus=[
            {
                'buttons': [
                    {
                        'args': [None, {'frame': {'duration': 100, 'redraw': True}, 'fromcurrent': True}],
                        'label': 'Play',
                        'method': 'animate'
                    },
                    {
                        'args': [[None], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                        'label': 'Pause',
                        'method': 'animate'
                    }
                ],
                'direction': 'left',
                'pad': {'r': 10, 't': 87},
                'showactive': True,
                'type': 'buttons',
                'x': 0.1,
                'xanchor': 'right',
                'y': 0,
                'yanchor': 'top'
            }
        ],
        sliders=[{
            'active': 0,
            'steps': [
                {
                    'label': str(i),
                    'method': 'animate',
                    'args': [
                        [str(i)],
                        {'mode': 'immediate', 'transition': {'duration': 300}}
                    ]
                }
                for i in range(500)
            ]
        }]
    )
)

# Add the frames to the figure
fig.frames = frames

fig.write_html('3dplot_NO_diffusion_noise.html')

# Show the figure
fig.show()